# resonansc marker and regulatory mechanism story

This notebook draws two story figures from `ResonanSC/outputs/result3`:

1. RNA-only marker expression: Healthy and COVID-19 are shown as two subplots. Each subplot is a `gene x cell type` heatmap. Marker genes are selected from the learned `M` matrix in `train_M_1.pt`, restricted to RNA gene-expression features, and scored to keep genes with similar cell-type patterns across conditions.
2. ATAC mapping mechanism: Healthy uses only HV ATAC batches, while COVID-19 uses only `COVID_SEV` and `COVID_MILD` ATAC batches. Peak-gene pairs are selected to favor within-condition batch consistency and between-condition mapping differences, then plotted at the individual-batch level.

Condition convention for `data/3/covid_rna.h5ad`: `G = Healthy`, `H = COVID-19`.

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
import seaborn as sns
import matplotlib.pyplot as plt
import torch

sns.set_theme(style="white", context="paper")
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
ROOT = Path("/home/jiayu/multimodal")
RESULT_DIR = ROOT / "ResonanSC/outputs/result3"
FIGURE_DIR = RESULT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_M = RESULT_DIR / "train_M_1.pt"
INIT_CKPT = RESULT_DIR / "init.pt"
INIT_ADATA = ROOT / "ResonanSC/data/processed/init_adata.h5ad"
COVID_RNA = ROOT / "data/3/covid_rna.h5ad"
COVID_ATAC = ROOT / "data/3/CBD-KEY-ATAC/CBD-KEY-ATAC.cell_by_peak.h5ad"

OUT_PREFIX = FIGURE_DIR / "result3_story"

TOP_GENES_PER_PROTO = 140
MAX_GENES_FOR_RNA_FIGURE = 40
MAX_ATAC_GENES = 12
PEAKS_PER_GENE = 4
MAX_ATAC_PEAKS = 36
MIN_PROTO_CELLS = 100
RNA_CELLTYPE_COL = "major_subset"
RNA_CONDITION_MAP = {"G": "Healthy", "H": "COVID-19"}
ATAC_COVID_CONDITIONS = {"COVID_SEV", "COVID_MILD"}
CELLTYPE_ORDER = ["CD4", "CD8", "MAIT", "NK", "B", "cMono", "ncMono", "DC", "PB"]

## Load trained M and label its columns by RNA cell type

In [ ]:
ckpt = torch.load(TRAIN_M, map_location="cpu", weights_only=False)
init_ckpt = torch.load(INIT_CKPT, map_location="cpu", weights_only=False)

gene_names = np.asarray(list(map(str, ckpt["gene_names"])), dtype=object)
M_prob = ckpt.get("M_prob", ckpt["M"]).detach().cpu().numpy()

init_adata = ad.read_h5ad(INIT_ADATA, backed="r")
rna_init_obs = init_adata.obs[init_adata.obs["batch"].astype(str).str.startswith("rna_")].copy()
rna_init_obs["pred_align_s"] = rna_init_obs["pred_align"].astype(str)

proto_rows = []
for proto in range(M_prob.shape[1]):
    sub = rna_init_obs[rna_init_obs["pred_align_s"] == str(proto)]
    if len(sub) == 0:
        proto_rows.append({"proto": proto, "celltype": f"proto_{proto}", "subtype": f"proto_{proto}", "n_cells": 0})
        continue
    proto_rows.append(
        {
            "proto": proto,
            "celltype": str(sub[RNA_CELLTYPE_COL].value_counts().idxmax()),
            "subtype": str(sub["subtype"].value_counts().idxmax()),
            "n_cells": len(sub),
        }
    )
proto_meta = pd.DataFrame(proto_rows)
init_adata.file.close()

valid_protos = proto_meta.query("n_cells >= @MIN_PROTO_CELLS")["proto"].tolist()
display(proto_meta)

## Select M marker genes with similar RNA cell-type patterns across conditions

In [ ]:
def to_dense(x):
    if sparse.issparse(x):
        return x.to_memory().toarray() if hasattr(x, "to_memory") else x.toarray()
    return np.asarray(x)

rna = ad.read_h5ad(COVID_RNA, backed="r")
gene_expression_features = set(
    rna.var_names[rna.var["feature_types"].astype(str).eq("Gene Expression")]
)

candidate_rows = []
for proto in valid_protos:
    score = M_prob[:, proto]
    order = np.argsort(score)[::-1][:TOP_GENES_PER_PROTO]
    celltype = proto_meta.loc[proto_meta["proto"] == proto, "celltype"].iloc[0]
    for gene_idx in order:
        gene = str(gene_names[gene_idx])
        if gene not in gene_expression_features:
            continue
        other = np.delete(M_prob[gene_idx, :], proto)
        candidate_rows.append(
            {
                "gene": gene,
                "gene_idx": int(gene_idx),
                "proto": int(proto),
                "celltype": celltype,
                "M_score": float(score[gene_idx]),
                "M_specificity": float(score[gene_idx] / (other.max() + 1e-6)),
            }
        )

candidate_markers = (
    pd.DataFrame(candidate_rows)
    .sort_values(["gene", "M_score"], ascending=[True, False])
    .drop_duplicates("gene")
)
candidate_markers = candidate_markers[candidate_markers["gene"].isin(rna.var_names)].copy()
candidate_genes = candidate_markers["gene"].tolist()

gene_idx = [int(rna.var_names.get_loc(gene)) for gene in candidate_genes]
x = to_dense(rna[:, gene_idx].X).astype(float)
x = np.log1p(x)

meta = rna.obs[["condition", RNA_CELLTYPE_COL]].copy()
meta["condition_group"] = meta["condition"].astype(str).map(RNA_CONDITION_MAP)
expr = pd.DataFrame(x, columns=candidate_genes)
expr_meta = pd.concat([meta.reset_index(drop=True), expr.reset_index(drop=True)], axis=1)
condition_celltype_means = expr_meta.groupby(
    ["condition_group", RNA_CELLTYPE_COL], observed=True
)[candidate_genes].mean()
rna.file.close()

celltypes = [
    ct for ct in CELLTYPE_ORDER
    if ("Healthy", ct) in condition_celltype_means.index
    and ("COVID-19", ct) in condition_celltype_means.index
]

score_rows = []
for _, row in candidate_markers.iterrows():
    gene = row["gene"]
    celltype = row["celltype"]
    if celltype not in celltypes:
        continue
    healthy = condition_celltype_means.loc[("Healthy", celltypes), gene].to_numpy(float)
    covid = condition_celltype_means.loc[("COVID-19", celltypes), gene].to_numpy(float)
    target_i = celltypes.index(celltype)
    healthy_off = np.max(np.delete(healthy, target_i))
    covid_off = np.max(np.delete(covid, target_i))
    specificity_min = min(healthy[target_i] - healthy_off, covid[target_i] - covid_off)
    condition_corr = np.corrcoef(healthy, covid)[0, 1] if np.std(healthy) > 0 and np.std(covid) > 0 else 0.0
    plot_score = float(row["M_score"]) * (max(specificity_min, 0.0) + 0.02) * (condition_corr + 1.0) / 2.0
    score_rows.append(
        {
            **row.to_dict(),
            "expr_target_min": min(healthy[target_i], covid[target_i]),
            "expr_specificity_min": specificity_min,
            "condition_corr": condition_corr,
            "plot_score": plot_score,
        }
    )

marker_scores = pd.DataFrame(score_rows)
selected_blocks = []
for celltype, block in marker_scores.query("expr_target_min > 0.003 and condition_corr > 0.10").groupby("celltype"):
    selected_blocks.append(block.sort_values("plot_score", ascending=False).head(6))

celltype_rank = {celltype: i for i, celltype in enumerate(CELLTYPE_ORDER)}
selected_markers = pd.concat(selected_blocks).sort_values("plot_score", ascending=False).head(MAX_GENES_FOR_RNA_FIGURE)
selected_markers = (
    selected_markers.assign(celltype_rank=lambda x: x["celltype"].map(celltype_rank).fillna(999).astype(int))
    .sort_values(["celltype_rank", "celltype", "plot_score"], ascending=[True, True, False])
    .drop(columns="celltype_rank")
    .reset_index(drop=True)
)
selected_genes = selected_markers["gene"].tolist()
selected_markers.to_csv(f"{OUT_PREFIX}_selected_M_gene_markers.csv", index=False)
display(selected_markers[["gene", "celltype", "proto", "M_score", "condition_corr", "plot_score"]])

## Figure 1: RNA marker expression, Healthy vs COVID-19

In [ ]:
def row_zscore(df):
    centered = df.sub(df.mean(axis=1), axis=0)
    scaled = centered.div(df.std(axis=1).replace(0, np.nan), axis=0)
    return scaled.fillna(0.0)

rna_mats = {}
for condition in ["Healthy", "COVID-19"]:
    mat = condition_celltype_means.loc[(condition, celltypes), selected_genes].T
    mat.index = selected_genes
    mat.columns = celltypes
    rna_mats[condition] = mat

combined = pd.concat([rna_mats["Healthy"], rna_mats["COVID-19"]], axis=1)
combined_z = row_zscore(combined)
healthy_z = combined_z.iloc[:, : len(celltypes)]
covid_z = combined_z.iloc[:, len(celltypes) :]

healthy_z.to_csv(f"{OUT_PREFIX}_rna_marker_heatmap_Healthy_zscore.csv")
covid_z.to_csv(f"{OUT_PREFIX}_rna_marker_heatmap_COVID_zscore.csv")

fig = plt.figure(figsize=(12.4, max(7, 0.22 * len(selected_genes) + 2)))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.045], wspace=0.08)
axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]
cbar_ax = fig.add_subplot(gs[0, 2])
RNA_TITLE_FONTSIZE = 14
RNA_LABEL_FONTSIZE = 12
RNA_TICK_FONTSIZE = 10

for ax, condition, mat in zip(axes, ["Healthy", "COVID-19"], [healthy_z, covid_z]):
    sns.heatmap(
        mat,
        ax=ax,
        cmap="vlag",
        center=0,
        cbar=(ax is axes[-1]),
        cbar_ax=cbar_ax if ax is axes[-1] else None,
        cbar_kws={"label": "row z-score of log1p mean RNA"} if ax is axes[-1] else None,
    )
    ax.set_title(condition, fontsize=RNA_TITLE_FONTSIZE)
    ax.set_xlabel("RNA cell type", fontsize=RNA_LABEL_FONTSIZE)
    ax.set_ylabel("M-selected gene marker" if ax is axes[0] else "", fontsize=RNA_LABEL_FONTSIZE)
    ax.tick_params(axis="x", rotation=45, labelsize=RNA_TICK_FONTSIZE)
    ax.tick_params(axis="y", labelsize=RNA_TICK_FONTSIZE)
axes[1].tick_params(axis="y", left=False, labelleft=False)
cbar_ax.tick_params(labelsize=RNA_TICK_FONTSIZE)
cbar_ax.set_ylabel("row z-score of log1p mean RNA", fontsize=RNA_LABEL_FONTSIZE)

fig.suptitle("resonansc M-selected RNA markers show similar cell-type patterns", y=1.02, fontsize=15)
fig.savefig(f"{OUT_PREFIX}_rna_M_gene_marker_by_celltype.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{OUT_PREFIX}_rna_M_gene_marker_by_celltype.pdf", bbox_inches="tight")
plt.show()

## Select condition-differential peak-gene mappings for the same marker genes

In [ ]:
atac = ad.read_h5ad(COVID_ATAC, backed="r")
atac_meta = atac.obs[["batch", "condition"]].drop_duplicates().copy()
atac.file.close()

atac_meta["mapping_batch"] = atac_meta["batch"].map(init_ckpt["atac_batch_mapping"])
atac_meta["condition_group"] = atac_meta["condition"].map(
    lambda x: "Healthy" if str(x) == "HV" else ("COVID-19" if str(x) in ATAC_COVID_CONDITIONS else "Other")
)
display(atac_meta.sort_values("mapping_batch"))

edge_rows = []
selected_gene_set = set(selected_genes)
for batch_id, weights in ckpt["mapping_weights"].items():
    info = ckpt["mapping_init"][batch_id]
    batch = str(info["batch_name"])
    peak_idx = info["peak_idx"].detach().cpu().numpy().astype(int)
    gene_idx = info["gene_idx"].detach().cpu().numpy().astype(int)
    learned_weight = weights.detach().cpu().numpy().astype(float)
    edge_genes = np.asarray(info["gene_names"], dtype=object)[gene_idx].astype(str)
    keep = np.isin(edge_genes, list(selected_gene_set))
    if not keep.any():
        continue
    edge_rows.append(
        pd.DataFrame(
            {
                "batch": batch,
                "peak": np.asarray(info["peak_names"], dtype=object)[peak_idx[keep]],
                "gene": edge_genes[keep],
                "weight": learned_weight[keep],
            }
        )
    )

edges = pd.concat(edge_rows, ignore_index=True)
edges = edges.merge(
    atac_meta[["mapping_batch", "condition", "condition_group"]].rename(columns={"mapping_batch": "batch"}),
    on="batch",
    how="left",
)
edges = edges[edges["condition_group"].isin(["Healthy", "COVID-19"])].copy()

def _batch_number(batch_name):
    return int(str(batch_name).split("_")[-1])

batch_plot_meta = atac_meta[atac_meta["condition_group"].isin(["Healthy", "COVID-19"])].copy()
batch_plot_meta["group_order"] = batch_plot_meta["condition_group"].map({"Healthy": 0, "COVID-19": 1})
batch_plot_meta["batch_order"] = batch_plot_meta["mapping_batch"].map(_batch_number)
batch_plot_meta = batch_plot_meta.sort_values(["group_order", "batch_order"]).reset_index(drop=True)
batch_order = batch_plot_meta["mapping_batch"].tolist()
batch_to_condition = dict(zip(batch_plot_meta["mapping_batch"], batch_plot_meta["condition_group"]))
healthy_batches = [batch for batch in batch_order if batch_to_condition[batch] == "Healthy"]
covid_batches = [batch for batch in batch_order if batch_to_condition[batch] == "COVID-19"]

mapping_by_condition = edges.groupby(
    ["condition_group", "peak", "gene"], observed=True
)["weight"].mean().reset_index()

pair_batch_wide = edges[edges["batch"].isin(batch_order)].pivot_table(
    index=["peak", "gene"], columns="batch", values="weight", fill_value=0
).reindex(columns=batch_order, fill_value=0)
pair_scores = pair_batch_wide.reset_index()
pair_scores["Healthy"] = pair_scores[healthy_batches].mean(axis=1)
pair_scores["COVID-19"] = pair_scores[covid_batches].mean(axis=1)
pair_scores["healthy_support"] = (pair_scores[healthy_batches] > 0).sum(axis=1)
pair_scores["covid_support"] = (pair_scores[covid_batches] > 0).sum(axis=1)
pair_scores["healthy_std"] = pair_scores[healthy_batches].std(axis=1).fillna(0)
pair_scores["covid_std"] = pair_scores[covid_batches].std(axis=1).fillna(0)
pair_scores["healthy_cv"] = pair_scores["healthy_std"] / (pair_scores["Healthy"] + 1e-6)
pair_scores["covid_cv"] = pair_scores["covid_std"] / (pair_scores["COVID-19"] + 1e-6)
pair_scores["diff_abs"] = (pair_scores["COVID-19"] - pair_scores["Healthy"]).abs()
pair_scores["max_weight"] = pair_scores[["Healthy", "COVID-19"]].max(axis=1)
pair_scores["dominant_condition"] = np.where(pair_scores["Healthy"] >= pair_scores["COVID-19"], "Healthy", "COVID-19")
pair_scores["dominant_support"] = np.where(
    pair_scores["dominant_condition"] == "Healthy", pair_scores["healthy_support"], pair_scores["covid_support"]
)
pair_scores["dominant_total_batches"] = np.where(
    pair_scores["dominant_condition"] == "Healthy", len(healthy_batches), len(covid_batches)
)
pair_scores["dominant_cv"] = np.where(
    pair_scores["dominant_condition"] == "Healthy", pair_scores["healthy_cv"], pair_scores["covid_cv"]
)
pair_scores["dominant_std"] = np.where(
    pair_scores["dominant_condition"] == "Healthy", pair_scores["healthy_std"], pair_scores["covid_std"]
)
pair_scores["other_mean"] = np.minimum(pair_scores["Healthy"], pair_scores["COVID-19"])
pair_scores["dominant_support_fraction"] = pair_scores["dominant_support"] / pair_scores["dominant_total_batches"]
pair_scores["diff_abs_batch_mean"] = pair_scores["diff_abs"]
pair_scores["within_std_sum"] = pair_scores["healthy_std"] + pair_scores["covid_std"]
pair_scores["support_count"] = pair_scores["healthy_support"] + pair_scores["covid_support"]
pair_scores["consistency_score"] = (
    pair_scores["diff_abs"]
    + 0.12 * pair_scores["dominant_support_fraction"]
    - 0.18 * pair_scores["dominant_cv"]
    - 0.08 * pair_scores["other_mean"]
)

candidate_pairs = pair_scores.query("max_weight > 0.05").copy()

pair_blocks = []
for gene, block in candidate_pairs.groupby("gene"):
    pair_blocks.append(block.sort_values(["diff_abs", "max_weight"], ascending=False).head(PEAKS_PER_GENE))
selected_pairs = pd.concat(pair_blocks).sort_values(["diff_abs", "max_weight"], ascending=False)

plot_genes = selected_pairs.groupby("gene")["diff_abs"].max().sort_values(ascending=False).head(MAX_ATAC_GENES).index.tolist()
selected_pairs = selected_pairs[selected_pairs["gene"].isin(plot_genes)].head(MAX_ATAC_PEAKS + 4)
plot_peaks = selected_pairs["peak"].drop_duplicates().head(MAX_ATAC_PEAKS).tolist()
plot_genes = [gene for gene in plot_genes if gene in set(selected_pairs["gene"])]
selected_pairs.to_csv(f"{OUT_PREFIX}_selected_peak_gene_pairs.csv", index=False)
display(selected_pairs.head(40))

## Figure 2: condition-level peak-gene mapping plus batch consistency summary

In [ ]:
mapping_mats = {}
for condition in ["Healthy", "COVID-19"]:
    mat = mapping_by_condition[mapping_by_condition["condition_group"] == condition].pivot_table(
        index="peak", columns="gene", values="weight", fill_value=0
    )
    mat = mat.reindex(index=plot_peaks, columns=plot_genes, fill_value=0)
    mapping_mats[condition] = mat

mapping_mats["Healthy"].to_csv(f"{OUT_PREFIX}_atac_mapping_HV_matrix.csv")
mapping_mats["COVID-19"].to_csv(f"{OUT_PREFIX}_atac_mapping_COVID_matrix.csv")

paired_rows = []
pair_lookup = selected_pairs.set_index(["peak", "gene"])
for gene in plot_genes:
    gene_pairs = selected_pairs[selected_pairs["gene"] == gene].sort_values("diff_abs", ascending=False)
    for _, row in gene_pairs.iterrows():
        peak = row["peak"]
        if peak not in plot_peaks:
            continue
        paired_rows.append(
            {
                "gene": gene,
                "peak": peak,
                "Healthy": float(mapping_mats["Healthy"].loc[peak, gene]) if gene in mapping_mats["Healthy"].columns else 0.0,
                "COVID-19": float(mapping_mats["COVID-19"].loc[peak, gene]) if gene in mapping_mats["COVID-19"].columns else 0.0,
                "diff_abs": float(pair_lookup.loc[(peak, gene), "diff_abs"]),
            }
        )

paired_df = pd.DataFrame(paired_rows)
paired_df["delta_COVID_minus_Healthy"] = paired_df["COVID-19"] - paired_df["Healthy"]
paired_df["max_weight"] = paired_df[["Healthy", "COVID-19"]].max(axis=1)
paired_df["row_label"] = paired_df["peak"].str.replace("chr", "", regex=False)
paired_df.to_csv(f"{OUT_PREFIX}_atac_mapping_paired_peak_gene_weights.csv", index=False)

gene_order = plot_genes
peak_order = plot_peaks
gene_to_x = {gene: i * 2 for i, gene in enumerate(gene_order)}
peak_to_y = {peak: i for i, peak in enumerate(peak_order)}
plot_long = paired_df.melt(
    id_vars=["gene", "peak", "row_label", "diff_abs", "max_weight"],
    value_vars=["Healthy", "COVID-19"],
    var_name="condition_group",
    value_name="mapping_weight",
)
plot_long = plot_long[plot_long["mapping_weight"] > 0].copy()
plot_long["x"] = plot_long["gene"].map(gene_to_x).astype(float)
plot_long["x"] += plot_long["condition_group"].map({"Healthy": 0.0, "COVID-19": 0.82})
plot_long["y"] = plot_long["peak"].map(peak_to_y).astype(float)
plot_long["condition_label"] = plot_long["condition_group"].map({"Healthy": "Healthy HV", "COVID-19": "COVID-19"})

DOT_TITLE_FONTSIZE = 15
DOT_LABEL_FONTSIZE = 13
DOT_TICK_FONTSIZE = 11
DOT_LEGEND_FONTSIZE = 11
condition_palette = {"Healthy HV": "#4c72b0", "COVID-19": "#c44e52"}
weight_max = plot_long["mapping_weight"].max()
plot_long["point_size"] = 28 + 155 * (plot_long["mapping_weight"] / weight_max).fillna(0)
plot_long.to_csv(f"{OUT_PREFIX}_atac_mapping_gene_condition_peak_dotmatrix_points.csv", index=False)

fig, ax = plt.subplots(figsize=(max(9.2, 0.68 * len(gene_order) + 2.2), max(6.8, 0.21 * len(peak_order) + 2.0)))
for label, block in plot_long.groupby("condition_label", sort=False):
    ax.scatter(
        block["x"],
        block["y"],
        s=block["point_size"],
        color=condition_palette[label],
        alpha=0.78,
        edgecolor="white",
        linewidth=0.45,
        label=label,
        zorder=3,
    )

for y in np.arange(len(peak_order)):
    ax.axhline(y, color="#f0f0f0", lw=0.7, zorder=0)
for i, gene in enumerate(gene_order):
    left = i * 2 - 0.36
    right = i * 2 + 1.18
    ax.axvspan(left, right, color="#fafafa" if i % 2 == 0 else "#ffffff", zorder=-1)
    if i > 0:
        ax.axvline(i * 2 - 0.58, color="#d8d8d8", lw=0.8, zorder=1)

gene_centers = [i * 2 + 0.41 for i in range(len(gene_order))]
ax.set_xticks(gene_centers)
ax.set_xticklabels(gene_order, rotation=45, ha="right", fontsize=DOT_TICK_FONTSIZE)
ax.set_yticks([])
ax.invert_yaxis()
ax.set_xlim(-0.75, (len(gene_order) - 1) * 2 + 1.55)
ax.set_ylim(len(peak_order) - 0.45, -1.75)
ax.set_xlabel("M-selected marker gene", fontsize=DOT_LABEL_FONTSIZE)
ax.set_ylabel("Peak", fontsize=DOT_LABEL_FONTSIZE)
ax.set_title("Condition-level peak-gene mapping weights for selected marker genes", fontsize=DOT_TITLE_FONTSIZE)
condition_legend = ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0), title="Condition", borderaxespad=0, fontsize=DOT_LEGEND_FONTSIZE, title_fontsize=DOT_LEGEND_FONTSIZE)
ax.add_artist(condition_legend)
size_values = np.linspace(plot_long["mapping_weight"].min(), weight_max, 3)
size_handles = [
    ax.scatter([], [], s=28 + 155 * (value / weight_max), color="#777777", alpha=0.65, edgecolor="white", linewidth=0.45)
    for value in size_values
]
ax.legend(
    size_handles,
    [f"{value:.2f}" for value in size_values],
    title="Mapping weight",
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.02, 0.72),
    scatterpoints=1,
    fontsize=DOT_LEGEND_FONTSIZE,
    title_fontsize=DOT_LEGEND_FONTSIZE,
)
fig.tight_layout()
fig.savefig(f"{OUT_PREFIX}_atac_mapping_gene_condition_peak_dotmatrix.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{OUT_PREFIX}_atac_mapping_gene_condition_peak_dotmatrix.pdf", bbox_inches="tight")
plt.show()

selected_pair_keys = paired_df[["peak", "gene"]].drop_duplicates()
selected_batch_vectors = pair_batch_wide.reset_index().merge(selected_pair_keys, on=["peak", "gene"], how="inner")
batch_vectors = selected_batch_vectors.set_index(["peak", "gene"])[batch_order].T
batch_condition = {batch: batch_to_condition[batch] for batch in batch_order}

def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return np.nan
    return float(np.dot(a, b) / denom)

condition_means = {
    "Healthy": batch_vectors.loc[healthy_batches].mean(axis=0).to_numpy(float),
    "COVID-19": batch_vectors.loc[covid_batches].mean(axis=0).to_numpy(float),
}
similarity_rows = []
for batch in batch_order:
    condition = batch_condition[batch]
    other_condition = "COVID-19" if condition == "Healthy" else "Healthy"
    vector = batch_vectors.loc[batch].to_numpy(float)
    similarity_rows.extend(
        [
            {
                "batch": batch,
                "condition": condition,
                "target": "Other condition mean",
                "cosine_similarity": cosine_similarity(vector, condition_means[other_condition]),
            },
            {
                "batch": batch,
                "condition": condition,
                "target": "Own condition mean",
                "cosine_similarity": cosine_similarity(vector, condition_means[condition]),
            },
        ]
    )

similarity_df = pd.DataFrame(similarity_rows).dropna()
similarity_df.to_csv(f"{OUT_PREFIX}_atac_mapping_batch_consistency_summary.csv", index=False)
summary_order = ["Other condition mean", "Own condition mean"]
target_to_x = {target: i for i, target in enumerate(summary_order)}
batch_condition_label = {"Healthy": "Healthy HV batch", "COVID-19": "COVID-19 batch"}
batch_palette = {"Healthy": "#4c72b0", "COVID-19": "#c44e52"}

SUMMARY_TITLE_FONTSIZE = 12
SUMMARY_LABEL_FONTSIZE = 10
SUMMARY_TICK_FONTSIZE = 10
SUMMARY_LEGEND_FONTSIZE = 9

fig, ax = plt.subplots(figsize=(5.4, 4.1))
for batch, block in similarity_df.groupby("batch", sort=False):
    block = block.set_index("target").loc[summary_order].reset_index()
    condition = block["condition"].iloc[0]
    ax.plot(
        [target_to_x[target] for target in block["target"]],
        block["cosine_similarity"],
        color=batch_palette[condition],
        alpha=0.55,
        lw=1.6,
        zorder=1,
    )
    ax.scatter(
        [target_to_x[target] for target in block["target"]],
        block["cosine_similarity"],
        color=batch_palette[condition],
        s=48,
        edgecolor="white",
        linewidth=0.5,
        zorder=2,
    )

for condition, color in batch_palette.items():
    ax.scatter([], [], color=color, s=58, label=batch_condition_label[condition])
ax.legend(frameon=False, loc="lower right", fontsize=SUMMARY_LEGEND_FONTSIZE)
ax.set_xticks([0, 1])
ax.set_xticklabels(summary_order, fontsize=SUMMARY_TICK_FONTSIZE)
ax.set_ylabel("Cosine similarity", fontsize=SUMMARY_LABEL_FONTSIZE)
ax.set_title("Each batch is closer to its own condition profile", fontsize=SUMMARY_TITLE_FONTSIZE)
ax.set_ylim(0, 1.05)
ax.tick_params(axis="y", labelsize=SUMMARY_TICK_FONTSIZE)
sns.despine(ax=ax)
fig.subplots_adjust(left=0.24, right=0.98, bottom=0.24, top=0.86)
fig.savefig(f"{OUT_PREFIX}_atac_mapping_batch_consistency_summary.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{OUT_PREFIX}_atac_mapping_batch_consistency_summary.pdf", bbox_inches="tight")
plt.show()